# Workflow for a multi-regional energy system

In this application of the ETHOS.FINE framework, a multi-regional energy system is modeled and optimized.

All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

The workflow is structures as follows:
1. Required packages are imported and the input data path is set
2. An energy system model instance is created
3. Commodity sources are added to the energy system model
4. Commodity conversion components are added to the energy system model
5. Commodity storages are added to the energy system model
6. Commodity transmission components are added to the energy system model
7. Commodity sinks are added to the energy system model
8. The energy system model is optimized
9. Selected optimization results are presented


# 1. Import required packages and set input data path

The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [ ]:
import fine as fn
from getData import getData
import pandas as pd
from pathlib import Path

cwd = Path.cwd()
data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# 2. Create an energy system model instance 

The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {
    "cluster_0",
    "cluster_1",
    "cluster_2",
    "cluster_3",
    "cluster_4",
    "cluster_5",
    "cluster_6",
    "cluster_7",
}
commodityUnitDict = {
    "electricity": r"GW$_{el}$",
    "methane": r"GW$_{CH_{4},LHV}$",
    "biogas": r"GW$_{biogas,LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    "hydrogen": r"GW$_{H_{2},LHV}$",
}
commodities = {"electricity", "hydrogen", "methane", "biogas", "CO2"}
numberOfTimeSteps = 8760
hoursPerTimeStep = 1
CO2_reductionTarget = 1
balanceLimit = pd.DataFrame(
    index=["CO2 limit"],
    columns=["Total", "lowerBound"],
    data=[[-366 * (1 - CO2_reductionTarget), True]],
)

In [3]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    balanceLimit=balanceLimit,
)

# 3. Add commodity sources to the energy system model

## 3.1. Electricity sources

### Wind onshore

In [4]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (onshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=1.1,
        opexPerCapacity=1.1 * 0.02,
        interestRate=0.08,
        economicLifetime=20,
    )
)

Full load hours:

In [5]:
data["Wind (onshore), operationRateMax"].sum()

cluster_0    1572.003960
cluster_1    2350.292663
cluster_2    2374.507270
cluster_3    2186.572278
cluster_4    1572.650655
cluster_5    1767.840650
cluster_6    2719.564564
cluster_7    1553.045964
dtype: float64

### Wind offshore

In [6]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (offshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (offshore), operationRateMax"],
        capacityMax=data["Wind (offshore), capacityMax"],
        investPerCapacity=2.3,
        opexPerCapacity=2.3 * 0.02,
        interestRate=0.08,
        economicLifetime=20,
    )
)

Full load hours:

In [7]:
data["Wind (offshore), operationRateMax"].sum()

cluster_0       0.000000
cluster_1    4435.420314
cluster_2    4301.655834
cluster_3    3902.391858
cluster_4       0.000000
cluster_5       0.000000
cluster_6    4609.508396
cluster_7       0.000000
dtype: float64

### PV

In [8]:
esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["PV, operationRateMax"],
        capacityMax=data["PV, capacityMax"],
        investPerCapacity=0.65,
        opexPerCapacity=0.65 * 0.02,
        interestRate=0.08,
        economicLifetime=25,
    )
)

Full load hours:

In [9]:
data["PV, operationRateMax"].sum()

cluster_0    1113.216464
cluster_1    1053.579422
cluster_2    1058.005181
cluster_3    1079.872237
cluster_4    1140.407380
cluster_5    1051.848141
cluster_6    1069.843344
cluster_7    1085.697466
dtype: float64

### Exisisting run-of-river hydroelectricity plants

In [10]:
esM.add(
    fn.Source(
        esM=esM,
        name="Existing run-of-river plants",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateFix=data["Existing run-of-river plants, operationRateFix"],
        tsaWeight=0.01,
        capacityFix=data["Existing run-of-river plants, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.208,
    )
)

## 3.2. Methane (natural gas and biogas)

### Natural gas

In [11]:
esM.add(
    fn.Source(
        esM=esM,
        name="Natural gas purchase",
        commodity="methane",
        hasCapacityVariable=False,
        commodityCost=0.0331 * 1e-3,
    )
)

### Biogas

In [12]:
esM.add(
    fn.Source(
        esM=esM,
        name="Biogas purchase",
        commodity="biogas",
        operationRateMax=data["Biogas, operationRateMax"],
        hasCapacityVariable=False,
        commodityCost=0.05409 * 1e-3,
    )
)

# 4. Add conversion components to the energy system model

### Combined cycle gas turbine plants

In [13]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "methane": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### New combined cycle gas turbine plants for biogas

In [14]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (biogas)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "biogas": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### New combined cycly gas turbines for hydrogen

In [15]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (hydrogen)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### Electrolyzers

In [16]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electrolyzer",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

### rSOC

In [17]:
capexRSOC = 1.5

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOEC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOFC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

# 5. Add commodity storages to the energy system model

## 5.1. Electricity storage

### Lithium ion batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [18]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

## 5.2. Hydrogen storage

### Hydrogen filled salt caverns
The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [19]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity=0.00011,
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.3. Methane storage

### Methane filled salt caverns

In [20]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (biogas)",
        commodity="biogas",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=443,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (methane), capacityMax"],
        investPerCapacity=0.00004,
        opexPerCapacity=0.00001,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.4 Pumped hydro storage

### Pumped hydro storage

In [21]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Pumped hydro storage",
        commodity="electricity",
        chargeEfficiency=0.88,
        dischargeEfficiency=0.88,
        hasCapacityVariable=True,
        selfDischarge=1 - (1 - 0.00375) ** (1 / (30 * 24)),
        chargeRate=0.16,
        dischargeRate=0.12,
        capacityFix=data["Pumped hydro storage, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.000153,
    )
)

# 6. Add commodity transmission components to the energy system model

## 6.1. Electricity transmission

### AC cables

esM.add(fn.LinearOptimalPowerFlow(esM=esM, name='AC cables', commodity='electricity',
                                  hasCapacityVariable=True, capacityFix=data['AC cables, capacityFix'],
                                  reactances=data['AC cables, reactances']))

In [22]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="AC cables",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityFix=data["AC cables, capacityFix"],
    )
)

### DC cables

In [23]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="DC cables",
        commodity="electricity",
        losses=data["DC cables, losses"],
        distances=data["DC cables, distances"],
        hasCapacityVariable=True,
        capacityFix=data["DC cables, capacityFix"],
    )
)

## 6.2 Methane transmission

### Methane pipeline

In [24]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (biogas)",
        commodity="biogas",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000037,
        investIfBuilt=0.000314,
        interestRate=0.08,
        economicLifetime=40,
    )
)

## 6.3 Hydrogen transmission

### Hydrogen pipelines

In [25]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (hydrogen)",
        commodity="hydrogen",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000177,
        investIfBuilt=0.00033,
        interestRate=0.08,
        economicLifetime=40,
    )
)

# 7. Add commodity sinks to the energy system model

## 7.1. Electricity sinks

### Electricity demand

In [26]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=data["Electricity demand, operationRateFix"],
    )
)

## 7.2. Hydrogen sinks

### Fuel cell electric vehicle (FCEV) demand

In [27]:
FCEV_penetration = 0.5
esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=data["Hydrogen demand, operationRateFix"] * FCEV_penetration,
    )
)

## 7.3. CO2 sinks

### CO2 exiting the system's boundary

In [28]:
esM.add(
    fn.Sink(
        esM=esM,
        name="CO2 to enviroment",
        commodity="CO2",
        hasCapacityVariable=False,
        balanceLimitID="CO2 limit",
    )
)

All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

# 8 Temporal Aggregation

In [29]:
esM.aggregateTemporally(numberOfTypicalPeriods=7)

C:\Users\l.wijesinghe\AppData\Local\Temp\ipykernel_9772\2487221595.py:1: DeprecationWarning: The keyword(s) ['numberOfTypicalPeriods'] belong to the ETHOS.TSAM 3.x interface, which is deprecated and will be removed in a future FINE release. Use the ETHOS.TSAM 4.x interface instead (numberOfTypicalPeriods -> n_clusters); see fine.aggregations.temporalAggregation.deprecatedKeywords for the full mapping.
  esM.aggregateTemporally(numberOfTypicalPeriods=7)


### Optimization with modelling to generate alternatives

In [ ]:
# The `optimizationSpecs` only work with the Gurobi solver. If you are using another solver you need to choose
# specs spcecific to this solver or no specs.

# # Check if pyM is populated after optimization
# Try to access objective if pyM is not None

try:
    print("Attempting to optimize model...")
    esM.optimize(
        timeSeriesAggregation=True,
        optimizationSpecs="OptimalityTol=1e-3 method=2 cuts=0 MIPGap=5e-3",
    )
    print("Optimization completed.")
    if esM.pyM is not None:
        obj_val = esM.pyM.Obj()
        print(f"Objective value: {obj_val}")

        results = fn.runMGA(
            esM,
            spec={
                "slack": 0.1,
                "objectives": [
                    {
                        "name": "iteration 1",
                        "sense": "minimize",
                        "variableSelection": "custom",
                        "terms": [
                            {
                                "component": "Wind (onshore)",
                                "variable": "capacity",
                                "weight": 0.3,
                            },
                            {"component": "PV", "variable": "operation", "weight": 0.6},
                            {
                                "component": "Pumped hydro storage",
                                "variable": "discharge",
                                "weight": 0.01,
                            },
                            {
                                "component": "AC cables",
                                "variable": "capacity",
                                "weight": 0.01,
                            },
                            {
                                "component": "DC cables",
                                "variable": "capacity",
                                "weight": 0.01,
                            },
                            {
                                "component": "Pipelines (biogas)",
                                "variable": "capacity",
                                "weight": 0.01,
                            },
                            {
                                "component": "Pipelines (hydrogen)",
                                "variable": "capacity",
                                "weight": 0.01,
                            },
                        ],
                    },
                    {
                        "name": "iteration 2",
                        "sense": "minimize",
                        "variableSelection": "custom",
                        "terms": [
                            {
                                "component": "Wind (onshore)",
                                "variable": "capacity",
                                "weight": 0.3,
                            },
                            {"component": "PV", "variable": "operation", "weight": 0.6},
                            {
                                "component": "Pumped hydro storage",
                                "variable": "discharge",
                                "weight": 0.9,
                            },
                            {
                                "component": "AC cables",
                                "variable": "capacity",
                                "weight": 0.8,
                            },
                            {
                                "component": "DC cables",
                                "variable": "capacity",
                                "weight": 0.4,
                            },
                            {
                                "component": "Pipelines (biogas)",
                                "variable": "capacity",
                                "weight": 0.67,
                            },
                            {
                                "component": "Pipelines (hydrogen)",
                                "variable": "capacity",
                                "weight": 0.12,
                            },
                        ],
                    },
                ],
            },
            save_mga_results_in_excel=True,
            excelOutputDir="mga_results",
            excelFilePrefix="MGA",
            excelKwargs=None,
        )
    else:
        print(
            "esM.pyM is None after optimization. Modeling to generate alternatives need an Objective Value"
        )
except Exception as e:
    print(f"ERROR during esM.optimize(): {e}")

Objective value: 62.62990306963728
MGA run: slack 10.0%, 2 alternative(s)
Excel output enabled: c:\Users\l.wijesinghe\Hackathon\FINE\examples\12_Modelling_to_Generate_Alternatives\mga_results
Accessing the cost-optimal solution ...
Cost-optimal solution found: total cost = 62.6299
  Writing results to Excel: mga_results\MGA_costOptimal.xlsx ...
  Excel file written (39.5 s).
Adding the near-optimal cost constraint ...
Declaring cost constraint: total cost <= 68.8929 (optimal cost 62.6299, slack 10.0%) ...
Cost constraint declared (0.0 s).
Solving MGA alternatives ...
----------------------------------------------------------------------
Alternative 1/2: 'itearion 1'
  Declaring MGA objective function (minimize, variableSelection='custom') ...
    Removing previous objective function ...
    Adding capacity of 'Wind (onshore)' (weight 0.3) ...
    Adding operation of 'PV' (weight 0.6) ...
    Adding discharge of 'Pumped hydro storage' (weight 0.01) ...
    Adding capacity of 'AC cables'

C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (biogas)
  warnings.warn(
C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(


  MGA objective value = 32.4777, total cost = 32.4777 (-48.14% vs. cost-optimal)
  Writing results to Excel: mga_results\MGA_itearion_1.xlsx ...
  Excel file written (27.3 s).
Alternative 'itearion 1' done (171.7 s).
----------------------------------------------------------------------
Alternative 2/2: 'itearion 2'
  Declaring MGA objective function (minimize, variableSelection='custom') ...
    Removing previous objective function ...
    Adding capacity of 'Wind (onshore)' (weight 0.3) ...
    Adding operation of 'PV' (weight 0.6) ...
    Adding discharge of 'Pumped hydro storage' (weight 0.9) ...
    Adding capacity of 'AC cables' (weight 0.8) ...
    Adding capacity of 'DC cables' (weight 0.4) ...
    Adding capacity of 'Pipelines (biogas)' (weight 0.67) ...
    Adding capacity of 'Pipelines (hydrogen)' (weight 0.12) ...
    Building objective expression from 1350 variables ...
  MGA objective function declared (0.1 s).
  Solving MGA problem ...
Set parameter Threads to value 3
Se

C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (biogas)
  warnings.warn(
C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(


  MGA objective value = 172.806, total cost = 33.48 (-46.54% vs. cost-optimal)
  Writing results to Excel: mga_results\MGA_itearion_2.xlsx ...
  Excel file written (32.5 s).
Alternative 'itearion 2' done (245.5 s).
MGA run finished in 456.8 s.


<!-- # 9. Modelling to Generate Alternatives -->